In [21]:
from chessdotcom import *
import pprint
import requests

import urllib.request
import json

import numpy as np
import pandas as pd
import pickle

In [22]:
from chessdotcom import Client

Client.request_config["headers"]["User-Agent"] = (
 "Machine learning for chess match outcome Prediction"
 "Contact me at conorjackvincent@live.co.uk"
)



#### Create two lists, One containing all titled GM's and another all titled WGM's on chess.com

In [23]:
def titled_player_names(title):
    data = get_titled_players(title)
    list = data.json["players"]
    return list

In [24]:
gm_list = titled_player_names("GM")
wgm_list = titled_player_names("WGM") 
im_list = titled_player_names("IM")
fm_list = titled_player_names("FM")
cm_list = titled_player_names("CM")
wim_list = titled_player_names("WIM")
wfm_list = titled_player_names("WFM")
wcm_list = titled_player_names("WCM")

In [25]:
print(f"Number of GMs: {len(gm_list)}   Number of WGMs: {len(wgm_list)}   Number of IMs: {len(im_list)}   Number of FMs: {len(fm_list)}   Number of CMs: {len(cm_list)} Number of WIMs: {len(wim_list)} Number of WFMs: {len(wfm_list)} Number of WCMs: {len(wcm_list)}")

# 1766 total

# 87.6% (GM, Males)
# 12.4% (WGM, Females)

Number of GMs: 1525   Number of WGMs: 216   Number of IMs: 2123   Number of FMs: 3455   Number of CMs: 1400 Number of WIMs: 372 Number of WFMs: 652 Number of WCMs: 349


In [26]:
# Create a DataFrame with a single column "usernames"

# gm_df = spark.createDataFrame([Row(Usernames=username) for username in gm_list])
# gm_df = gm_df.withColumn("title", F.lit("GM"))

# wgm_df = spark.createDataFrame([Row(Usernames=username) for username in wgm_list])
# wgm_df = wgm_df.withColumn("title", F.lit("WGM"))

# df = gm_df.union(wgm_df)

# df.sample(0.01).show()

In [27]:
# Show the DataFrame

# df.groupBy("title").count().show()

#### For each name in the two lists, lets get additional details about them!

In [28]:
player_profile = get_player_profile("azikom")
print(player_profile)

ChessDotComResponse(player=Collection(avatar='https://images.chesscomfiles.com/uploads/v1/user/44058794.2e18aa97.200x200o.854e2adae7a6.jpeg', player_id=44058794, id='https://api.chess.com/pub/player/azikom', url='https://www.chess.com/member/azikom', name='Azer Mirzoev', username='azikom', title='GM', followers=134, country='https://api.chess.com/pub/country/AZ', last_online=1706207253, joined=1520509648, status='premium', is_streamer=False, verified=False, league='Bronze'))


In [29]:
def build_df_player_profiles(username_list):
    """
    Build a list of player profiles based on the given list of usernames.

    Parameters:
    - username_list (list): A list of chess.com usernames.

    Returns:
    - final_list: A list of player profiles, each represented as a list of information (name, username, title, followers, 
            country_code, country, status, is_streamer, verified, league).
    - fail_list (list): A list of usernames for which profile retrieval failed after 15 retries.
    """

    # List to store lists of data records, username and player profile information in each list.
    final_list = []

    # List to store any usernames for which player profiles cannot be collected.
    fail_list = []

    # Counter used for visual purposes, showing each iteration of the below for loop.
    counter = 0

    # If a player profile cannot be obtained, this counter is used to stop the while loop
    retry = 0

    
    # For each username in the username_list
    for username in username_list:
        while True:
            try:
                # increment the counter and print the counter, overwriting the previous counter
                counter += 1
                print(f'\r{counter} {username}', end='', flush=True)
        
                # Get player profile response from chessdotcom.get_player_profile
                player_profile_response = get_player_profile(username)
                
                # Extract all relevant information, if not available, append None.
                player_info = player_profile_response.player
                name = getattr(player_info, 'name', None)
                username = getattr(player_info, 'username', None)
                title = getattr(player_info, 'title', None)
                followers = getattr(player_info, 'followers', None)
                status = getattr(player_info, 'status', None)
                is_streamer = getattr(player_info, 'is_streamer', None)
                verified = getattr(player_info, 'verified', None)
                league = getattr(player_info, 'league', None)
        
                country_code = ''
                country = ''
                with urllib.request.urlopen(player_info.country) as url:
                    data = json.load(url)
                    country_code = data['code']
                    country = data['name']

                # Create a list with all the relevant player profile information and append this list to the final_list.
                list_data = [name, username, title, followers, country_code, country, status, is_streamer, verified, league]
                final_list.append(list_data)
                break
                
            except ChessDotComError as e:
                print(f'\rError for {username}: {e}', flush=True)
                if retry < 15:
                    retry += 1
                    print(f'\rRetrying for {username} attempt {retry}...', end='', flush=True)
                else:
                    print(f'\rMax retries reached for {username}. Moving on.', flush=True)
                    fail_list.append(username)
                    retry = 0
                    break
            
    return final_list, fail_list

In [30]:
list1, list1_failed = build_df_player_profiles(wgm_list)
# df1 = df1.union(df2)

216 zvaigznite3_2ianer9

In [31]:
list2, list2_failed = build_df_player_profiles(gm_list)

1525 zvonokchess1996eyovaalla

In [32]:
list3, list3_failed = build_df_player_profiles(im_list)

2123 zydraxx1uchavaamazan3ile

In [33]:
list4, list4_failed = build_df_player_profiles(fm_list)

Error for elpayasodios: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"An internal error has occurred. Please contact Chess.com Developer's Forum for further help https://www.chess.com/club/chess-com-developer-community ."})
Error for elpayasodios: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"An internal error has occurred. Please contact Chess.com Developer's Forum for further help https://www.chess.com/club/chess-com-developer-community ."})
Error for elpayasodios: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"An internal error has occurred. Please contact Chess.com Developer's Forum for further help https://www.chess.com/club/chess-com-developer-community ."})
Error for elpayasodios: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"An internal error has occurred. Please contact Chess.com Developer's Forum for further hel

In [34]:
list5, list5_failed = build_df_player_profiles(cm_list)

1400 zverr1lev201012layer09y

In [35]:
list6, list6_failed = build_df_player_profiles(wim_list)

372 zcorrales5p097642rin8

In [36]:
list7, list7_failed = build_df_player_profiles(wfm_list)

652 zuzanak3afigullinaa1n

In [37]:
list8, list8_failed = build_df_player_profiles(wcm_list)

Error for kapibara_warri0r: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"An internal error has occurred. Please contact Chess.com Developer's Forum for further help https://www.chess.com/club/chess-com-developer-community ."})
Error for kapibara_warri0r: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"An internal error has occurred. Please contact Chess.com Developer's Forum for further help https://www.chess.com/club/chess-com-developer-community ."})
Error for kapibara_warri0r: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"An internal error has occurred. Please contact Chess.com Developer's Forum for further help https://www.chess.com/club/chess-com-developer-community ."})
Error for kapibara_warri0r: <class 'chessdotcom.types.ChessDotComError'>(status_code=404, text={"code":0,"message":"An internal error has occurred. Please contact Chess.com Developer's Forum

In [40]:
column_names = ["name", "username", "title", "followers", "country_code", "country", "status", "is_streamer", "verified", "league"]
plist1 = pd.DataFrame(list1, columns=column_names)
plist2 = pd.DataFrame(list2, columns=column_names)
plist3 = pd.DataFrame(list3, columns=column_names)
plist4 = pd.DataFrame(list4, columns=column_names)
plist5 = pd.DataFrame(list5, columns=column_names)
plist6 = pd.DataFrame(list6, columns=column_names)
plist7 = pd.DataFrame(list7, columns=column_names)
plist8 = pd.DataFrame(list8, columns=column_names)

data_frames = [plist1, plist2, plist3, plist4, plist5, plist6, plist7, plist8]
first_frame = pd.concat(data_frames)

df_elements = first_frame.sample(n=10)
df_elements.head(10)

,name,username,title,followers,country_code,country,status,is_streamer,verified,league
286,None,kiddoo55,WFM,0,XX,International,premium,False,False,Bronze
1529,Dionisio Aldama,queburumba,IM,12,US,United States,premium,False,False,None
1404,M Nubairshah Shaikh,nubairyt,IM,120,IN,India,premium,False,False,Elite
1923,Aniruddha Deshpande,malhar9,FM,113,IN,India,premium,False,False,Wood
2416,Barnabas Persanyi,persanyibarnabas,FM,0,HU,Hungary,premium,False,False,None
1865,Renier Castellanos,thinkingsquares,IM,159,ES,Spain,premium,False,False,Champion
847,Luis Liévano Alvarado,mavericksv,CM,15,SV,El Salvador,premium,False,False,Legend
145,None,juditova,WIM,4,US,United States,premium,False,False,Crystal
2792,Akshaya Kalaiyalahan,shaya01,FM,21,XE,England,premium,False,False,Crystal
633,Jesus Arturo Silva Hernández,jarshh95,CM,93,MX,Mexico,premium,False,False,Legend


In [41]:
first_frame.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10089 entries, 0 to 347
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          7948 non-null   object
 1   username      10089 non-null  object
 2   title         10089 non-null  object
 3   followers     10089 non-null  int64 
 4   country_code  10089 non-null  object
 5   country       10089 non-null  object
 6   status        10089 non-null  object
 7   is_streamer   10089 non-null  bool  
 8   verified      10089 non-null  bool  
 9   league        8342 non-null   object
dtypes: bool(2), int64(1), object(7)
memory usage: 729.1+ KB


In [42]:
second_frame = first_frame.dropna(subset=['league'])
second_frame.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8342 entries, 0 to 346
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          6629 non-null   object
 1   username      8342 non-null   object
 2   title         8342 non-null   object
 3   followers     8342 non-null   int64 
 4   country_code  8342 non-null   object
 5   country       8342 non-null   object
 6   status        8342 non-null   object
 7   is_streamer   8342 non-null   bool  
 8   verified      8342 non-null   bool  
 9   league        8342 non-null   object
dtypes: bool(2), int64(1), object(7)
memory usage: 602.8+ KB


In [43]:
with open('player_profiles_original.pkl', 'wb') as file:
    pickle.dump(first_frame, file)

with open('player_profiles_dropna.pkl', 'wb') as file:
    pickle.dump(second_frame, file)


In [31]:
with open('player_profiles_original.pkl', 'rb') as file:
    loaded_first_frame = pickle.load(file)

with open('player_profiles_dropna.pkl', 'rb') as file:
    loaded_second_frame = pickle.load(file)

In [32]:
loaded_first_frame.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8703 entries, 0 to 1395
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          6857 non-null   object
 1   username      8703 non-null   object
 2   title         8703 non-null   object
 3   followers     8703 non-null   int64 
 4   country_code  8703 non-null   object
 5   country       8703 non-null   object
 6   status        8703 non-null   object
 7   is_streamer   8703 non-null   bool  
 8   verified      8703 non-null   bool  
 9   league        7225 non-null   object
dtypes: bool(2), int64(1), object(7)
memory usage: 628.9+ KB


In [33]:
loaded_second_frame.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7225 entries, 0 to 1395
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          5741 non-null   object
 1   username      7225 non-null   object
 2   title         7225 non-null   object
 3   followers     7225 non-null   int64 
 4   country_code  7225 non-null   object
 5   country       7225 non-null   object
 6   status        7225 non-null   object
 7   is_streamer   7225 non-null   bool  
 8   verified      7225 non-null   bool  
 9   league        7225 non-null   object
dtypes: bool(2), int64(1), object(7)
memory usage: 522.1+ KB


In [ ]:
import urllib.request, json 
with urllib.request.urlopen("https://api.chess.com/pub/country/AZ") as url:
    data = json.load(url)
    print(data)
    print(data['code'])
    print(data['name'])


In [ ]:
printer = pprint.PrettyPrinter()

In [ ]:
def print_leaderboards():
    data = get_leaderboards().json
    categories = data.keys()

    for category in categories:
        ldrboard = category
    
    for category in data[ldrboard]:
        print('category', category)
        for idx, entry in enumerate(data[ldrboard][category]):
            print(f"Rank: {idx + 1} | Username: {entry['username']} | Rating: {entry['score']}")




def get_most_recent_game(username):
    data = get_player_game_archives(username).json

    url = data['archives'][-1]
    print(url)
    games = requests.get(url)
    print(games)

print_leaderboards()

In [ ]:
get_most_recent_game("Castlecard")

In [ ]:
temp_dict = {
    "authority" : "api.chess.com",
    ":method:"  : "GET",
    ":path:" : "/pub/player/castlecard/games/2023/11",
    ":scheme:" : "https"
}

hello = requests.get("https://api.chess.com/pub/player/castlecard/games/2023/11", params=temp_dict)
hello.content